In [ ]:
import os
import re
import gc
import sys
import math
import time
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict

import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

import warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("hymt2-srt-inference")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16

@dataclass
class CFG:
    model_name: str = "tencent/Hy-MT2-1.8B"
    output_dir: str = r"translated_subtitles"
    data_root: str = r"wmt26_video_subtitle_translation_testset"
    adapter_path: str = r"hymt2_qlora_out/best"
    tgt_lang: str = "English"
    batch_size: int = 16
    eval_gen_max_new: int = 160
    eval_gen_num_beams: int = 4

cfg = CFG()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)

def parse_srt_file(file_path: Path) -> List[Dict]:
    try:
        content = file_path.read_text(encoding="utf-8-sig", errors="ignore")
    except Exception as e:
        log.error(f"Failed to read file {file_path.name}: {e}")
        return []
    content = content.replace("\r\n", "\n").strip()
    raw_blocks = re.split(r'\n\n+', content)
    blocks = []
    for raw_block in raw_blocks:
        lines = [line.strip() for line in raw_block.split('\n') if line.strip()]
        if len(lines) >= 3:
            idx = lines[0]
            timestamp = lines[1]
            text = " ".join(lines[2:])
            blocks.append({"idx": idx, "timestamp": timestamp, "text": text})
        elif len(lines) == 2:
            idx = lines[0]
            timestamp = lines[1]
            blocks.append({"idx": idx, "timestamp": timestamp, "text": " "})
    return blocks

def write_srt_file(file_path: Path, blocks: List[Dict]):
    with open(file_path, "w", encoding="utf-8") as f:
        for b in blocks:
            f.write(f"{b['idx']}\n")
            f.write(f"{b['timestamp']}\n")
            f.write(f"{b['text']}\n\n")

PROMPT_TEMPLATE = """Translate the following text into {tgt_lang}. Note that you must ONLY output the translated result without any additional explanation:{src}"""

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def build_input_ids(src: str) -> List[int]:
    user_content = PROMPT_TEMPLATE.format(
        tgt_lang=cfg.tgt_lang,
        src=src
    )
    msgs = [{"role": "user", "content": user_content}]
    prompt_ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)
    return prompt_ids

class SRTFileDataset(Dataset):
    def __init__(self, blocks: List[Dict]):
        self.blocks = blocks

    def __len__(self):
        return len(self.blocks)
        
    def __getitem__(self, idx):
        b = self.blocks[idx]
        src_text = b["text"].strip() if b["text"] else " "
        if src_text:
            input_ids = build_input_ids(src_text)
            is_empty = False
        else:
            input_ids = build_input_ids(" ")
            is_empty = True
        return {
            "input_ids": input_ids,
            "idx": b["idx"],
            "timestamp": b["timestamp"],
            "is_empty": is_empty
        }

def collate_srt(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    ids = torch.full((len(batch), maxlen), pad, dtype=torch.long)
    attn = torch.zeros((len(batch), maxlen), dtype=torch.long)
    for i, b in enumerate(batch):
        L = len(b["input_ids"])
        pad_left = maxlen - L
        ids[i, pad_left:] = torch.tensor(b["input_ids"], dtype=torch.long)
        attn[i, pad_left:] = 1
    return {
        "input_ids": ids,
        "attention_mask": attn,
        "idx": [b["idx"] for b in batch],
        "timestamp": [b["timestamp"] for b in batch],
        "is_empty": [b["is_empty"] for b in batch]
    }

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

log.info("Loading base Hy-MT2 model...")
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_cfg,
    device_map={"": 0} if torch.cuda.is_available() else "auto",
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
    attn_implementation="sdpa",
)
base_model.config.use_cache = True
base_model.config.pad_token_id = tokenizer.pad_token_id

adapter_path = Path(cfg.adapter_path)
if adapter_path.exists():
    log.info(f"Mounting fine-tuned adapter from: {adapter_path}")
    model = PeftModel.from_pretrained(base_model, str(adapter_path))
else:
    log.warning(f"No adapter found at {adapter_path}. Running with the un-tuned base model!")
    model = base_model

model.eval()

src_path = Path(cfg.data_root)
if not src_path.exists():
    raise FileNotFoundError(f"Source folder not found: {src_path}")

srt_files = sorted(list(src_path.glob("*_zh.srt")))

log.info(f"Discovered {len(srt_files)} SRT files for translation.")

for srt_file in tqdm(srt_files, desc="Translating Files"):
    vid = srt_file.name.replace("_zh.srt", "")
    out_file_name = f"{vid}_en.srt"
    out_path = Path(cfg.output_dir) / out_file_name
    
    blocks = parse_srt_file(srt_file)
    if not blocks:
        log.warning(f"Skipping empty/unparseable file: {srt_file.name}")
        continue
        
    dataset = SRTFileDataset(blocks)
    loader = DataLoader(
        dataset, 
        batch_size=cfg.batch_size, 
        shuffle=False, 
        collate_fn=collate_srt,
        num_workers=0
    )
    
    translated_blocks = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attn = batch["attention_mask"].to(DEVICE)
            
            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attn,
                max_new_tokens=cfg.eval_gen_max_new,
                num_beams=cfg.eval_gen_num_beams,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True
            )
            
            for i in range(gen.shape[0]):
                is_empty = batch["is_empty"][i]
                orig_idx = batch["idx"][i]
                timestamp = batch["timestamp"][i]
                
                if is_empty:
                    cleaned_txt = ""
                else:
                    prompt_len = input_ids[i].shape[0]
                    new_tokens = gen[i, prompt_len:]
                    txt = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
                    cleaned_txt = txt.split("\n")[0].strip()
                
                translated_blocks.append({
                    "idx": orig_idx,
                    "timestamp": timestamp,
                    "text": cleaned_txt
                })
                
    write_srt_file(out_path, translated_blocks)

log.info(f"All SRT files successfully translated and saved to {cfg.output_dir}!")